# 🗑️ Untrashify — E-Waste Detection Training

**EfficientDet-D5 | 78 classes | COCO | head-only, batch=4**

### ⚠️ Before running
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. `ewaste_model.tar.gz` auto-downloads at the end

⏱️ **~15–20 min** on T4 (frozen backbone + batch=4 + 3 epochs)

## 0️⃣ Imports & Constants

In [ ]:
import os, sys, json, shutil, zipfile, urllib.request, glob, re, tarfile
from pathlib import Path
import torch

ROOT           = Path('/content')
REPO_DIR       = ROOT / 'Yet-Another-EfficientDet-Pytorch'
DATASET_DIR    = REPO_DIR / 'datasets' / 'ewaste'
CHECKPOINT_DIR = ROOT / 'ewaste_checkpoints'
LOG_DIR        = ROOT / 'ewaste_logs'
WEIGHTS_PATH   = REPO_DIR / 'weights' / 'efficientdet-d5.pth'
ONNX_PATH      = ROOT / 'ewaste_model.onnx'
CLASSES_PATH   = ROOT / 'ewaste_classes.json'
TAR_PATH       = ROOT / 'ewaste_model.tar.gz'

ROBOFLOW_API_KEY   = 'J5X5BymTiaxtbKzjd80N'
ROBOFLOW_WORKSPACE = 'electronic-waste-detection'
ROBOFLOW_PROJECT   = 'e-waste-dataset-r0ojc'
ROBOFLOW_VERSION   = 44

WEIGHTS_URL  = 'https://github.com/zylo117/Yet-Another-EfficientDet-Pytorch/releases/download/1.0/efficientdet-d5.pth'
REPO_ZIP_URL = 'https://github.com/zylo117/Yet-Another-EfficientDet-Pytorch/archive/refs/heads/master.zip'

print('✅ Imports OK')
print(f'   REPO_DIR       = {REPO_DIR}')
print(f'   CHECKPOINT_DIR = {CHECKPOINT_DIR}')

## 1️⃣ Verify GPU

In [ ]:
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime → Change runtime type → T4 GPU')

## 2️⃣ Install Dependencies

In [ ]:
%%capture
# Note: tensorboardX ≠ tensorboard — train.py needs tensorboardX
!pip install roboflow pycocotools webcolors tensorboard tensorboardX onnx onnxruntime-gpu

## 3️⃣ Download EfficientDet Repository

In [ ]:
if not (REPO_DIR / 'train.py').exists():
    zip_path = ROOT / 'efficientdet.zip'
    print('Downloading repo...')
    urllib.request.urlretrieve(REPO_ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(ROOT)
    extracted = next(p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith('Yet-Another'))
    if extracted != REPO_DIR:
        shutil.move(str(extracted), str(REPO_DIR))
    zip_path.unlink(missing_ok=True)
    print('✅ Repo downloaded')
else:
    print('✅ Repo already present')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

req = REPO_DIR / 'requirements.txt'
if req.exists():
    os.system(f'pip install -q -r "{req}"')
print('✅ Done')

## 4️⃣ Download Dataset

In [ ]:
from roboflow import Roboflow

if (DATASET_DIR / 'train' / '_annotations.coco.json').exists():
    print('✅ Dataset already present')
else:
    TMP_DIR = ROOT / 'rf_tmp'
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    print('Downloading dataset from Roboflow...')
    rf   = Roboflow(api_key=ROBOFLOW_API_KEY)
    proj = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

    _cwd = os.getcwd()
    os.chdir(TMP_DIR)
    proj.version(ROBOFLOW_VERSION).download('coco')
    os.chdir(_cwd)

    candidates = [d for d in TMP_DIR.iterdir() if d.is_dir()]
    if not candidates:
        raise RuntimeError('Roboflow download produced no folder')
    dl_dir = max(candidates, key=lambda d: d.stat().st_mtime)

    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.move(str(dl_dir), str(DATASET_DIR))
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    print('✅ Dataset ready')

IMG_EXTS = {'.jpg', '.jpeg', '.png'}
for split in ['train', 'valid', 'test']:
    d = DATASET_DIR / split
    if d.exists():
        n = sum(1 for f in d.iterdir() if f.suffix.lower() in IMG_EXTS)
        print(f'  {split:6s}: {n:,} images')

## 5️⃣ Reorganise Dataset & Extract Class Names

In [ ]:
ANN_DIR = DATASET_DIR / 'annotations'
ANN_DIR.mkdir(parents=True, exist_ok=True)

for split in ['train', 'valid']:
    src = DATASET_DIR / split / '_annotations.coco.json'
    dst = ANN_DIR / f'instances_{split}.json'
    if not src.exists():
        raise FileNotFoundError(f'Missing {src}')
    if not dst.exists():
        shutil.copy(src, dst)
        print(f'✅ Copied {split} annotations')
    else:
        print(f'✅ {dst.name} already exists')

with open(ANN_DIR / 'instances_train.json', encoding='utf-8') as f:
    _ann = json.load(f)
class_names = [c['name'] for c in sorted(_ann['categories'], key=lambda x: x['id'])]
print(f'\n📦 {len(class_names)} classes:', ', '.join(class_names[:8]), '...')

## 6️⃣ Write ewaste.yml Config

In [ ]:
PROJECT_DIR = REPO_DIR / 'projects'
PROJECT_DIR.mkdir(exist_ok=True)

valid_split = 'valid' if (DATASET_DIR / 'valid').exists() else 'val'

yaml_lines = [
    'project_name: ewaste',
    'train_set: train',
    f'val_set: {valid_split}',
    'num_gpus: 1',
    f'num_classes: {len(class_names)}',
    'compound_coef: 5',
    '',
    'mean: [0.485, 0.456, 0.406]',
    'std: [0.229, 0.224, 0.225]',
    '',
    "anchors_scales: '[2 ** 0, 2 ** (1.0 / 3.0), 2 ** (2.0 / 3.0)]'",
    "anchors_ratios: '[(1.0, 1.0), (1.4, 0.7), (0.7, 1.4)]'",
    '',
    'obj_list:',
] + [f"  - '{n}'" for n in class_names]

(PROJECT_DIR / 'ewaste.yml').write_text('\n'.join(yaml_lines) + '\n', encoding='utf-8')
print(f'✅ Config writen — {len(class_names)} classes, val_set={valid_split}')

## 7️⃣ Download Pretrained Weights

In [ ]:
WEIGHTS_PATH.parent.mkdir(exist_ok=True)
if WEIGHTS_PATH.exists():
    print(f'✅ Weights present ({WEIGHTS_PATH.stat().st_size / 1e6:.0f} MB)')
else:
    print('Downloading pretrained EfficientDet-D5 weights (~240 MB)...')
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print(f'✅ Saved ({WEIGHTS_PATH.stat().st_size / 1e6:.0f} MB)')

## 8️⃣ Patch train.py for PyTorch 2.x

In [ ]:
train_py = REPO_DIR / 'train.py'
src = train_py.read_text(encoding='utf-8')
patched = re.sub(
    r'ReduceLROnPlateau\(([^)]*),\s*verbose\s*=\s*True([^)]*)\)',
    r'ReduceLROnPlateau(\1\2)',
    src,
)
if patched != src:
    train_py.write_text(patched, encoding='utf-8')
    print('✅ Patched train.py')
else:
    print('✅ train.py already compatible')

## 9️⃣ Create Output Directories

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f'✅ Checkpoints → {CHECKPOINT_DIR}')
print(f'✅ Logs        → {LOG_DIR}')

## 🚀 10. Run Training

> **`--head_only True`** freezes the EfficientNet backbone → only BiFPN + head weights are updated
>
> Frozen backbone frees ~60% VRAM, so **batch=4 fits** and throughput doubles vs batch=2
>
> ⏱️ **~15–20 min** total on T4

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
# head_only=True  → backbone frozen, ~60% less VRAM → batch=4 fits on T4 15 GB
# 3 epochs        → enough to adapt the detection head to 78 e-waste classes
# LR=1e-3         → correct starting LR when only the head is being trained
NUM_EPOCHS  = 3
BATCH_SIZE  = 4
LR          = 1e-3
NUM_WORKERS = 2

cmd = (
    'cd /content/Yet-Another-EfficientDet-Pytorch && python train.py'
    ' -c 5'
    ' -p ewaste'
    f' --batch_size {BATCH_SIZE}'
    f' --lr {LR}'
    f' --num_epochs {NUM_EPOCHS}'
    ' --load_weights weights/efficientdet-d5.pth'
    ' --saved_path /content/ewaste_checkpoints'
    ' --log_path /content/ewaste_logs'
    f' --num_workers {NUM_WORKERS}'
    ' --head_only True'  # ← freezes backbone; DO NOT set to False on T4
)

print(f'Training: {NUM_EPOCHS} epochs, batch={BATCH_SIZE}, head_only=True')
!{cmd}

found = sorted(glob.glob('/content/ewaste_checkpoints/*.pth'))
print(f'\n✅ {len(found)} checkpoint(s) saved:')
for p in found:
    print(f'   {p}  ({Path(p).stat().st_size / 1e6:.0f} MB)')

## 📦 11. Export → ONNX  *(runs on CPU, no OOM)*

In [ ]:
import os, sys, json, glob, torch
from pathlib import Path

ROOT           = Path('/content')
REPO_DIR       = ROOT / 'Yet-Another-EfficientDet-Pytorch'
CHECKPOINT_DIR = ROOT / 'ewaste_checkpoints'
DATASET_DIR    = REPO_DIR / 'datasets' / 'ewaste'
ONNX_PATH      = ROOT / 'ewaste_model.onnx'

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU freed: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from backbone import EfficientDetBackbone

ann = DATASET_DIR / 'annotations' / 'instances_train.json'
with open(ann, encoding='utf-8') as f:
    class_names = [c['name'] for c in sorted(json.load(f)['categories'], key=lambda x: x['id'])]
print(f'{len(class_names)} classes')

PATTERNS = [
    str(CHECKPOINT_DIR / 'efficientdet-d5_*.pth'),
    str(CHECKPOINT_DIR / 'ewaste_*.pth'),
    str(CHECKPOINT_DIR / '*.pth'),
    '/content/**/*.pth',
]
checkpoints = []
for pat in PATTERNS:
    found = [p for p in sorted(glob.glob(pat, recursive=True))
             if not p.endswith('efficientdet-d5.pth')]
    if found:
        print(f'Checkpoint found: {pat}')
        checkpoints = found
        break

if not checkpoints:
    all_pth = glob.glob('/content/**/*.pth', recursive=True)
    print('❌ No checkpoint. All .pth files:')
    for p in all_pth: print(f'  {p}')
    raise RuntimeError('Re-run Cell 10.')

ckpt_path = checkpoints[-1]
print(f'Using: {ckpt_path}')

model = EfficientDetBackbone(
    num_classes=len(class_names), compound_coef=5,
    ratios=[(1.0,1.0),(1.4,0.7),(0.7,1.4)],
    scales=[2**0, 2**(1/3), 2**(2/3)],
)
state = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(state)
model.eval().cpu()
del state

dummy = torch.randn(1, 3, 1024, 1024)
torch.onnx.export(
    model, dummy, str(ONNX_PATH), opset_version=12,
    input_names=['input'],
    output_names=['features','regression','classification','anchors'],
    dynamic_axes={'input': {0: 'batch'}},
)
print(f'\n✅ ONNX: {ONNX_PATH}  ({ONNX_PATH.stat().st_size/1e6:.0f} MB)')

## 💾 12. Save Class Names

In [ ]:
import json
from pathlib import Path

CLASSES_PATH = Path('/content/ewaste_classes.json')
ANN_FILE = Path('/content/Yet-Another-EfficientDet-Pytorch/datasets/ewaste/annotations/instances_train.json')

with open(ANN_FILE, encoding='utf-8') as f:
    class_names = [c['name'] for c in sorted(json.load(f)['categories'], key=lambda x: x['id'])]

with open(CLASSES_PATH, 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=2)

print(f'✅ {len(class_names)} classes saved → {CLASSES_PATH}')

## 📥 13. Package & Download

In [ ]:
import tarfile
from pathlib import Path
from google.colab import files

CHECKPOINT_DIR = Path('/content/ewaste_checkpoints')
ONNX_PATH      = Path('/content/ewaste_model.onnx')
CLASSES_PATH   = Path('/content/ewaste_classes.json')
TAR_PATH       = Path('/content/ewaste_model.tar.gz')

with tarfile.open(TAR_PATH, 'w:gz') as tar:
    for ckpt in sorted(CHECKPOINT_DIR.glob('*.pth')):
        tar.add(ckpt, arcname=f'checkpoints/{ckpt.name}')
        print(f'  + checkpoints/{ckpt.name}')
    if ONNX_PATH.exists():
        tar.add(ONNX_PATH, arcname='ewaste_model.onnx')
        print('  + ewaste_model.onnx')
    else:
        print('  ⚠️  ewaste_model.onnx missing — run Cell 11')
    if CLASSES_PATH.exists():
        tar.add(CLASSES_PATH, arcname='ewaste_classes.json')
        print('  + ewaste_classes.json')

print(f'\n✅ {TAR_PATH}  ({TAR_PATH.stat().st_size/1e6:.0f} MB)')
files.download(str(TAR_PATH))

## ✅ Done!

| File | Purpose |
|---|---|
| `checkpoints/efficientdet-d5_*.pth` | PyTorch checkpoint |
| `ewaste_model.onnx` | ONNX for deployment |
| `ewaste_classes.json` | 78 class labels |

**Locally:** extract → copy files to `training/` → run `python training/infer_demo.py` → `streamlit run app_streamlit.py`

---
### 📊 Optional: TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/ewaste_logs